In [1]:
%matplotlib inline
import sys, os, json
import torch
import torch.nn as nn
import numpy as np
import torchvision
from torch.utils.data import DataLoader

sys.path.append('..')
from src.data.degredation import get_transforms
from src.models.resnet import resnet18  # CIFAR-native 32x32
from src.training.train import train, test

import matplotlib.pyplot as plt
from custom.figure import mm, color

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
plt.rcParams['font.family'] = 'DejaVu Sans'


In [ ]:
dir = "cifar10-native-gradual-lr"
num_net = 10
output_size = 10
lr_target = 0.0001  # normal-phase Adam lr 

figure_dir = os.path.join("..", "figures", dir)
save_dir = os.path.join("..", "results", dir)
warmup_dir = os.path.join("..", "results", "cifar10-native-0919")  # r warm-up checkpoints
os.makedirs(figure_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

criterion = nn.CrossEntropyLoss()
batch_size = 128
num_workers = 4

dataset_dir = "../dataset"
train_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=True, download=True,
    transform=get_transforms("cifar10", blur=0, color=1, test=False))
train_degradation_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=True, download=True,
    transform=get_transforms("cifar10", blur=7, color=0, test=False))
test_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=False, download=True,
    transform=get_transforms("cifar10", blur=0, color=1, test=True))
test_degradation_dataset = torchvision.datasets.CIFAR10(
    root=dataset_dir, train=False, download=True,
    transform=get_transforms("cifar10", blur=7, color=0, test=True))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,
                           num_workers=num_workers, persistent_workers=True)
train_degradation_loader = DataLoader(train_degradation_dataset, batch_size=batch_size, shuffle=True,
                                       num_workers=num_workers, persistent_workers=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False,
                          num_workers=num_workers, persistent_workers=True)
test_degradation_loader = DataLoader(test_degradation_dataset, batch_size=batch_size, shuffle=False,
                                      num_workers=num_workers, persistent_workers=True)

print(f"Train: {len(train_dataset)}  Train(degraded): {len(train_degradation_dataset)}  "
      f"Test: {len(test_dataset)}  Test(degraded): {len(test_degradation_dataset)}")


c:\Users\vslab#1\Desktop\Chaewon\critical-period\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Train: 50000  Train(degraded): 50000  Test: 10000  Test(degraded): 10000


In [ ]:
def warmup_ckpt_path(net_idx, r=10):
    return os.path.join(warmup_dir, f"warmup_native_lr0.1_r{r}_{net_idx}.pth")

def ckpt_path(tag, i):
    return os.path.join(save_dir, f"best_model_{tag}_{i}.pth")

def info_path(tag, i):
    return os.path.join(save_dir, f"training_info_{tag}_{i}.json")

def run_gl_pairs(tag_prefix, gl_pairs, r=10, lr_high=0.01, lr_low=lr_target, num_net=num_net, target_total=None):
    """Train with blur images for g epochs, then clean images for l epochs, for each (g, l) pair."""
    results = {}

    for g, l in gl_pairs:
        n_epochs = max(g, l, target_total or 0)
        tag = f"{tag_prefix}_g{g}l{l}"
        net_infos = []

        for net_idx in range(num_net):
            info = None
            start_epoch = 0
            if os.path.exists(ckpt_path(tag, net_idx)) and os.path.exists(info_path(tag, net_idx)):
                with open(info_path(tag, net_idx)) as f:
                    info = json.load(f)
                start_epoch = len(info["train_loss"])

            if info is not None and start_epoch >= n_epochs:
                print(f"[{tag}] net {net_idx} already done ({start_epoch} epochs), loaded from disk")
                net_infos.append(info)
                continue

            model = resnet18(num_classes=output_size).to(device)
            if info is not None:
                model.load_state_dict(torch.load(ckpt_path(tag, net_idx), map_location=device))
                print(f"[{tag}] net {net_idx} resuming from epoch {start_epoch}")
            else:
                warmup_path = warmup_ckpt_path(net_idx, r)
                assert os.path.exists(warmup_path), f"missing warm-up checkpoint: {warmup_path}"
                model.load_state_dict(torch.load(warmup_path, map_location=device))
                info = dict(train_loss=[], train_acc=[], blur_test_loss=[], blur_test_acc=[],
                            clean_test_loss=[], clean_test_acc=[], lr=[], image=[])

            optimizer = torch.optim.Adam(model.parameters(), lr=lr_high)

            for epoch in range(start_epoch, n_epochs):
                current_lr = lr_high if epoch < l else lr_low
                for grp in optimizer.param_groups:
                    grp["lr"] = current_lr

                loader = train_degradation_loader if epoch < g else train_loader
                image_type = "blur" if epoch < g else "clean"

                train_loss, train_acc = train(model, loader, optimizer, criterion, device=device)
                blur_test_loss, blur_test_acc = test(model, test_degradation_loader, criterion, device=device)
                clean_test_loss, clean_test_acc = test(model, test_loader, criterion, device=device)

                info["train_loss"].append(train_loss); info["train_acc"].append(train_acc)
                info["blur_test_loss"].append(blur_test_loss); info["blur_test_acc"].append(blur_test_acc)
                info["clean_test_loss"].append(clean_test_loss); info["clean_test_acc"].append(clean_test_acc)
                info["lr"].append(current_lr)
                info["image"].append(image_type)

                print(f"[{tag}] net {net_idx} epoch {epoch+1}/{n_epochs} lr={current_lr} image={image_type} "
                      f"train_acc={train_acc:.4f} blur_test_acc={blur_test_acc:.4f} clean_test_acc={clean_test_acc:.4f}")

            torch.save(model.state_dict(), ckpt_path(tag, net_idx))
            with open(info_path(tag, net_idx), "w") as f:
                json.dump(info, f)
            net_infos.append(info)

        results[(g, l)] = net_infos

    return results


In [ ]:
gl_pairs = [
    (5, 1), (5, 5), (5, 10),
    (10, 1), (10, 10), (10, 20),
    (20, 1), (20, 20), (20, 30),
    (40, 40),   # l=40 the whole time, all-high-lr 
    (50, 50),   # l=50 the whole time, all-high-lr
    (30, 10),   # g>l: 10 epochs high-lr, then 20 epochs low-lr
    (30, 20),   # g>l: 20 epochs high-lr, then 10 epochs low-lr
]

gl_results = run_gl_pairs("gradual_r10", gl_pairs, r=10)


[gradual_r10_g5l1] net 0 already done, loaded from disk
[gradual_r10_g5l1] net 1 already done, loaded from disk
[gradual_r10_g5l1] net 2 already done, loaded from disk
[gradual_r10_g5l1] net 3 already done, loaded from disk
[gradual_r10_g5l1] net 4 already done, loaded from disk
[gradual_r10_g5l1] net 5 already done, loaded from disk
[gradual_r10_g5l1] net 6 already done, loaded from disk
[gradual_r10_g5l1] net 7 already done, loaded from disk
[gradual_r10_g5l1] net 8 already done, loaded from disk
[gradual_r10_g5l1] net 9 already done, loaded from disk
[gradual_r10_g5l5] net 0 already done, loaded from disk
[gradual_r10_g5l5] net 1 already done, loaded from disk
[gradual_r10_g5l5] net 2 already done, loaded from disk
[gradual_r10_g5l5] net 3 already done, loaded from disk
[gradual_r10_g5l5] net 4 already done, loaded from disk
[gradual_r10_g5l5] net 5 already done, loaded from disk
[gradual_r10_g5l5] net 6 already done, loaded from disk
[gradual_r10_g5l5] net 7 already done, loaded fr

남은 전체 학습 with fine image, original lr 

In [ ]:
gl_finetune_results = run_gl_pairs("gradual_r10", gl_pairs, r=10, target_total=100)


compare with lr_start = 0.001, 0.01 

In [ ]:
gl_results_lr001 = run_gl_pairs("gradual_r10_lr001", gl_pairs, r=10, lr_high=0.001)


In [ ]:
gl_finetune_results_lr001 = run_gl_pairs("gradual_r10_lr001", gl_pairs, r=10, lr_high=0.001, target_total=100)


Summary: final blur/clean test accuracy for each (g, l) pair.

In [5]:
print(f"{'(g,l)':<10}{'epochs':<8}{'blur_acc':<12}{'std':<10}{'clean_acc':<12}{'std':<10}")
for g, l in gl_pairs:
    infos = gl_results[(g, l)]
    blur_accs = [info["blur_test_acc"][-1] for info in infos]
    clean_accs = [info["clean_test_acc"][-1] for info in infos]
    print(f"g{g}l{l:<7}{max(g, l):<8}"
          f"{np.mean(blur_accs):<12.4f}{np.std(blur_accs):<10.4f}"
          f"{np.mean(clean_accs):<12.4f}{np.std(clean_accs):<10.4f}")


(g,l)     epochs  blur_acc    std       clean_acc   std       
g5l1      5       0.4953      0.0214    0.4340      0.0182    
g5l5      5       0.6729      0.0086    0.6044      0.0290    
g5l10     10      0.2247      0.0230    0.8348      0.0068    
g10l1      10      0.5184      0.0214    0.4412      0.0201    
g10l10     10      0.7361      0.0061    0.6724      0.0180    
g10l20     20      0.2360      0.0207    0.8659      0.0056    
g20l1      20      0.5740      0.0138    0.4719      0.0226    
g20l20     20      0.7749      0.0045    0.7241      0.0199    
g20l30     30      0.3448      0.0410    0.8547      0.0041    
g40l40     40      0.7796      0.0040    0.7485      0.0174    
g50l50     50      0.7811      0.0051    0.7506      0.0225    
g30l10     30      0.7827      0.0035    0.7275      0.0121    
g30l20     30      0.8031      0.0037    0.7631      0.0047    
